# Part 1 — Tasks 2 & 3: Descriptive Statistics, Correlation and Probability

Smart City Traffic Intelligence capstone. Dataset: Metro Interstate Traffic Volume,
48,204 hourly records for westbound I-94, Oct 2012 – Sep 2018.

Part 1 analyses the data **as supplied**. Cleaning happens in Part 2, so where a
data-quality issue affects a result it is reported as a limitation rather than fixed here.


In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

REPO_ROOT = Path.cwd().parents[1]
DB = REPO_ROOT / "data" / "processed" / "traffic.db"

con = sqlite3.connect(DB)
df = pd.read_sql("SELECT * FROM traffic", con, parse_dates=["date_time"])
con.close()

print(f"{len(df):,} rows x {df.shape[1]} columns")
df.head()


48,204 rows x 9 columns


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,None,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,None,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


## Task 2.1 — Traffic volume statistics


In [2]:
vol = df["traffic_volume"]

stats_table = pd.Series({
    "count":              len(vol),
    "mean":               vol.mean(),
    "median":             vol.median(),
    "standard deviation": vol.std(ddof=1),
    "variance":           vol.var(ddof=1),
    "minimum":            vol.min(),
    "maximum":            vol.max(),
    "range":              vol.max() - vol.min(),
})
stats_table.round(2)


count                   48204.00
mean                     3259.82
median                   3380.00
standard deviation       1986.86
variance              3947615.32
minimum                     0.00
maximum                  7280.00
range                    7280.00
dtype: float64

In [3]:
# Quartiles and skewness, to support the interpretation below.
print(vol.quantile([0.25, 0.50, 0.75]))
print(f"\nskewness: {vol.skew():.3f}")
print(f"coefficient of variation: {vol.std(ddof=1) / vol.mean():.3f}")
print(f"hours recording exactly zero volume: {(vol == 0).sum()}")


0.25    1193.0
0.50    3380.0
0.75    4933.0
Name: traffic_volume, dtype: float64

skewness: -0.089
coefficient of variation: 0.610
hours recording exactly zero volume: 2


### Interpretation

Mean hourly volume is **3,259.82** vehicles and the median is **3,380** — the median sits
above the mean, and skewness is **−0.089**, so the distribution is very slightly
left-skewed and close to symmetric.

Variability is the more important result. The standard deviation is **1,986.86**, a
coefficient of variation of **0.61**: one standard deviation is roughly 61% of the mean.
The interquartile range runs from **1,193** to **4,933**, and the full range spans
**0 to 7,280**. Hourly volume on this corridor is not usefully described by its average —
the data is close to bimodal, reflecting the split between overnight hours and daytime
peaks rather than random scatter around a central value.

The practical consequence for the mobility team: any capacity or staffing decision based
on the mean will be badly wrong at both ends of the day. Hour-of-day is the variable that
matters, which is why it becomes a core feature in Parts 2 and 3.

Two rows record a volume of exactly **0**, which is physically implausible for a major
interstate and is treated as a sensor artefact in Part 2.


## Task 2.2 — Correlation between temperature and traffic volume


In [4]:
# The raw data contains 10 rows with temp = 0 K (absolute zero: sensor failure).
# Report the correlation both ways so the effect of those rows is explicit.
n_zero_k = (df["temp"] == 0).sum()
print(f"rows with temp = 0 K: {n_zero_k}")

r_raw, p_raw = stats.pearsonr(df["temp"], df["traffic_volume"])
valid = df[df["temp"] > 0]
r_val, p_val = stats.pearsonr(valid["temp"], valid["traffic_volume"])
rho = valid["temp"].corr(valid["traffic_volume"], method="spearman")

print(f"\nPearson r  (all rows)      : {r_raw:.4f}")
print(f"Pearson r  (temp > 0 K)    : {r_val:.4f}   n = {len(valid):,}   p = {p_val:.3g}")
print(f"Spearman rho (temp > 0 K)  : {rho:.4f}")
print(f"r-squared                  : {r_val**2:.4f}")


rows with temp = 0 K: 10

Pearson r  (all rows)      : 0.1303
Pearson r  (temp > 0 K)    : 0.1323   n = 48,194   p = 4.71e-187
Spearman rho (temp > 0 K)  : 0.1326
r-squared                  : 0.0175


In [5]:
# Correlation across the other numeric variables, for context.
df[["temp", "rain_1h", "snow_1h", "clouds_all", "traffic_volume"]].corr().round(3)


,temp,rain_1h,snow_1h,clouds_all,traffic_volume
temp,1.000,0.009,-0.020,-0.102,0.130
rain_1h,0.009,1.000,-0.000,0.005,0.005
snow_1h,-0.020,-0.000,1.000,0.028,0.001
clouds_all,-0.102,0.005,0.028,1.000,0.067
traffic_volume,0.130,0.005,0.001,0.067,1.000


### Interpretation

**Direction.** Pearson's r is **+0.132** (temp > 0 K), so the relationship is positive:
warmer hours are associated with slightly higher traffic volume.

**Strength.** The relationship is weak. r² is **0.018**, meaning temperature accounts for
under 2% of the variation in hourly traffic volume. The remaining 98% is driven by other
factors — overwhelmingly time of day. Spearman's rho (**0.133**) is nearly identical to
Pearson's r, so the weakness is not an artefact of a non-linear relationship that a linear
measure is missing; the association is genuinely slight.

The p-value is effectively zero (**4.7e-187**), but with n = 48,194 that is expected and
carries no practical weight: at this sample size a trivially small correlation is still
statistically significant. Significance and importance are different questions, and here
the effect size is what matters.

The 10 sensor-failure rows barely move the coefficient (0.1303 raw vs 0.1323 filtered),
because 10 rows out of 48,204 cannot shift a correlation appreciably. They are still
excluded on principle — a temperature of absolute zero is not data.

**Why correlation does not imply causation.** The positive association almost certainly
reflects a **common cause** rather than temperature driving traffic. Both variables follow
a daily cycle: temperature peaks in the afternoon, and so does traffic. Both also follow a
seasonal cycle. Hour of day drives both, producing a correlation between them without any
causal link — warmer air does not put cars on the road. The correlation is also
**confounded by season** (summer brings both heat and holiday travel patterns) and could
in principle run the other way, since dense traffic itself generates heat. Establishing
causation would require controlling for hour and season, which is exactly what the
regression models in Part 3 do.


## Task 3 — Probability and congestion analysis

Congestion is defined as `traffic_volume > 5500`. "Clear weather" uses
`weather_main = 'Clear'`; "high temperature" is `temp > 292 K` (about 18.9 °C).


In [6]:
CONGESTION_THRESHOLD = 5500
HIGH_TEMP_K = 292.0

congested = df["traffic_volume"] > CONGESTION_THRESHOLD
clear     = df["weather_main"] == "Clear"
high_temp = df["temp"] > HIGH_TEMP_K

n = len(df)
p_congestion = congested.mean()
p_clear      = clear.mean()
p_both       = (congested & clear).mean()

print(f"n = {n:,}\n")
print(f"P(Congestion)                = {p_congestion:.4f}   ({congested.sum():,} hours)")
print(f"P(Clear Weather)             = {p_clear:.4f}   ({clear.sum():,} hours)")
print(f"P(Congestion AND Clear)      = {p_both:.4f}   ({(congested & clear).sum():,} hours)")


n = 48,204

P(Congestion)                = 0.1473   (7,100 hours)
P(Clear Weather)             = 0.2778   (13,391 hours)
P(Congestion AND Clear)      = 0.0366   (1,763 hours)


### 3.2 Conditional probability


In [7]:
p_clear_given_cong = (congested & clear).sum() / congested.sum()
p_hot_given_cong   = (congested & high_temp).sum() / congested.sum()

print(f"P(Clear | Congestion)        = {p_clear_given_cong:.4f}")
print(f"P(High Temp | Congestion)    = {p_hot_given_cong:.4f}")
print(f"\nFor comparison, unconditional P(Clear) = {p_clear:.4f}")
print(f"                unconditional P(High Temp) = {high_temp.mean():.4f}")


P(Clear | Congestion)        = 0.2483
P(High Temp | Congestion)    = 0.2630

For comparison, unconditional P(Clear) = 0.2778
                unconditional P(High Temp) = 0.2438


### Independence test:  P(A and B) = P(A) x P(B) ?


In [8]:
expected_if_independent = p_congestion * p_clear
difference = p_both - expected_if_independent

print(f"P(Congestion AND Clear) observed        = {p_both:.4f}")
print(f"P(Congestion) x P(Clear) if independent = {expected_if_independent:.4f}")
print(f"difference                              = {difference:+.4f}")
print(f"ratio observed/expected                 = {p_both / expected_if_independent:.4f}")

# Chi-squared test on the 2x2 contingency table as a formal check.
table = pd.crosstab(congested, clear)
chi2, p_chi, dof, expected = stats.chi2_contingency(table)
print(f"\nchi-squared = {chi2:.1f}, p = {p_chi:.3g}")
table


P(Congestion AND Clear) observed        = 0.0366
P(Congestion) x P(Clear) if independent = 0.0409
difference                              = -0.0043
ratio observed/expected                 = 0.8938

chi-squared = 35.9, p = 2.06e-09


weather_main,False,True
traffic_volume,,
False,29476,11628
True,5337,1763


### Odds ratio: congestion in clear versus cloudy weather


In [9]:
clear_rows  = df[df["weather_main"] == "Clear"]
cloudy_rows = df[df["weather_main"] == "Clouds"]

a = (clear_rows["traffic_volume"]  > CONGESTION_THRESHOLD).sum()   # clear,  congested
b = len(clear_rows) - a                                            # clear,  not congested
c = (cloudy_rows["traffic_volume"] > CONGESTION_THRESHOLD).sum()   # cloudy, congested
d = len(cloudy_rows) - c                                           # cloudy, not congested

odds_clear  = a / b
odds_cloudy = c / d
odds_ratio  = odds_clear / odds_cloudy

print(f"Clear  : {a:,} congested of {len(clear_rows):,}  -> odds {odds_clear:.4f}")
print(f"Cloudy : {c:,} congested of {len(cloudy_rows):,}  -> odds {odds_cloudy:.4f}")
print(f"\nOdds ratio (clear vs cloudy) = {odds_ratio:.4f}")
print(f"Reciprocal (cloudy vs clear) = {1/odds_ratio:.4f}")


Clear  : 1,763 congested of 13,391  -> odds 0.1516
Cloudy : 2,592 congested of 15,164  -> odds 0.2062

Odds ratio (clear vs cloudy) = 0.7354
Reciprocal (cloudy vs clear) = 1.3598


### Interpretation

**Congestion is uncommon.** Volume exceeds 5,500 in **14.7%** of recorded hours (7,100 of
48,204). Clear weather covers **27.8%** of hours, and the two coincide in **3.66%**.

**Conditioning on congestion barely changes the weather picture.** P(Clear | Congestion)
is **0.2483**, against an unconditional P(Clear) of **0.2778**. Knowing that an hour is
congested makes clear weather slightly *less* likely, not more — a shift of about three
percentage points.

**The two events are not independent, but the dependence is weak and negative.** The
independence test gives P(Congestion) × P(Clear) = **0.0409** against an observed
**0.0366**; the observed joint probability is **89%** of what independence predicts. The
chi-squared test rejects independence, but again, with n = 48,204 that rejection is almost
guaranteed — the size of the departure is what matters, and it is small.

**The odds ratio points the same way.** Congestion occurs in 1,763 of 13,391 clear hours
(odds 0.1516) and 2,592 of 15,164 cloudy hours (odds 0.2062), giving an odds ratio of
**0.735**. The odds of congestion in clear weather are about **26% lower** than in cloudy
weather — equivalently, congestion is about **1.36 times** more likely when cloudy.

**What this suggests, and the trap to avoid.** Read naively, this says good weather
reduces congestion, which is implausible as a causal claim. The likely explanation is
again confounding by time and season. Clear skies are more common overnight and in winter;
cloud cover is more common during daytime hours and in transitional seasons. Because
congestion is overwhelmingly a daytime phenomenon, any weather category that is
over-represented at night will show artificially low congestion odds. The weather variable
is partly acting as a proxy for time of day.

The operational conclusion for the mobility team is therefore that **weather is a poor
standalone predictor of congestion on this corridor**. Weather effects should be assessed
only after conditioning on hour and day type, which is the approach taken in the modelling
in Part 3.
